# News Headline Generator: fine-tuning t5-small on XSum
Runs on JupyterLab. A CUDA GPU (8GB+ VRAM) is strongly recommended. Run cells top to bottom.

## 1. Environment setup and dataset loading

In [ ]:
%pip install -q -U transformers datasets evaluate rouge_score accelerate sentencepiece
# PyTorch must already be installed with CUDA support (https://pytorch.org/get-started/locally/)

In [ ]:
import os
import numpy as np
import torch
from datasets import load_dataset

USE_CUDA = torch.cuda.is_available()
if USE_CUDA:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no CUDA GPU found. Training will be very slow; consider a much smaller TRAIN_SUBSET.")

# Everything is saved relative to the notebook's folder
OUTPUT_DIR = "./t5-small-xsum-headlines"
SAVE_DIR = "./t5-headline-final"

raw = load_dataset("EdinburghNLP/xsum")
print(raw)
print(raw["train"][0])   # keys: document, summary, id

If loading fails with a "dataset scripts are no longer supported" error, run `%pip install -q "datasets<4.0"`,
restart the runtime, and use `load_dataset("EdinburghNLP/xsum", trust_remote_code=True)`.

In [ ]:
TRAIN_SUBSET = 60_000   # raise if your session allows; None = full dataset
EVAL_SUBSET = 1_000

train_raw = raw["train"].shuffle(seed=42)
if TRAIN_SUBSET:
    train_raw = train_raw.select(range(TRAIN_SUBSET))
eval_raw = raw["validation"].shuffle(seed=42).select(range(EVAL_SUBSET))
test_raw = raw["test"].shuffle(seed=42).select(range(500))

## 2. Preprocessing

In [ ]:
from transformers import AutoTokenizer

MODEL_CKPT = "t5-small"
MAX_SOURCE_LEN = 512
MAX_TARGET_LEN = 64
PREFIX = "summarize: "

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def preprocess(batch):
    inputs = [PREFIX + doc for doc in batch["document"]]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LEN,
        truncation=True,          # no padding here; the collator pads dynamically per batch
    )
    labels = tokenizer(
        text_target=batch["summary"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

cols = raw["train"].column_names
train_ds = train_raw.map(preprocess, batched=True, remove_columns=cols, desc="Tokenizing train")
eval_ds  = eval_raw.map(preprocess,  batched=True, remove_columns=cols, desc="Tokenizing eval")
test_ds  = test_raw.map(preprocess,  batched=True, remove_columns=cols, desc="Tokenizing test")

## 3. Model and data collator

In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CKPT)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,                # lets the collator build decoder_input_ids from labels
    label_pad_token_id=-100,    # padded label positions are ignored by the loss
    pad_to_multiple_of=8,       # better tensor-core utilization under fp16
)

## 4. Metrics, training arguments, trainer

In [ ]:
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
    decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )
    result = {k: round(v * 100, 2) for k, v in result.items()}
    result["gen_len"] = round(float(np.mean(
        [np.count_nonzero(p != tokenizer.pad_token_id) for p in preds]
    )), 1)
    return result

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # --- OOM-safe batching: effective batch = 8 x 4 = 32 ---
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    # gradient_checkpointing=True,   # uncomment if you raise batch size / lengths and hit OOM (slower)

    # --- Optimization ---
    learning_rate=3e-4,              # T5 tolerates higher LRs than BERT-style models
    weight_decay=0.01,
    num_train_epochs=2,
    lr_scheduler_type="linear",
    warmup_ratio=0.03,
    max_grad_norm=1.0,
    fp16=USE_CUDA,                   # if loss turns NaN, set fp16=False (t5-small fits in fp32)
    # bf16=True,                     # use instead of fp16 on Ampere+ GPUs (RTX 30xx/40xx, A100)

    # --- Eval / generation (kept cheap) ---
    eval_strategy="steps",
    eval_steps=500,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    generation_num_beams=1,          # greedy during training evals; use beams at final inference

    # --- Checkpointing ---
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    logging_steps=50,
    dataloader_num_workers=0 if os.name == "nt" else 2,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    processing_class=tokenizer,      # named `tokenizer=` in older transformers versions
    compute_metrics=compute_metrics,
)

## 5. Train, evaluate, save

In [ ]:
trainer.train()
# To resume after a disconnect: trainer.train(resume_from_checkpoint=True)

In [ ]:
# Final test metrics with beam search
trainer.args.generation_num_beams = 4
print(trainer.evaluate(test_ds, metric_key_prefix="test"))

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

## 6. Sanity check

In [ ]:
def headline(article, num_beams=4):
    inputs = tokenizer(PREFIX + article, return_tensors="pt",
                       max_length=MAX_SOURCE_LEN, truncation=True).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, num_beams=num_beams, max_new_tokens=48,
                             no_repeat_ngram_size=3, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

sample = raw["test"][3]
print("PRED:", headline(sample["document"]))
print("REF :", sample["summary"])

## Done
The model is saved in `./t5-headline-final`. Keep that folder next to `app.py` and run the Flask server from a JupyterLab terminal.